# LightGCN — ml-1m Training Notebook (High Precision)

**Dataset**: MovieLens ml-1m (1M ratings packed into just 3,700 movies ~ VERY DENSE)  
**Why this dataset?**: Density is required for GNNs to learn proper neighborhood embeddings. This guarantees high-precision similarity.
**Output**: `embeddings.pt` + `movieid_to_tmdbid.json` → drop both into `backend/`

## Before running
1. Download **ml-1m** from https://files.grouplens.org/datasets/movielens/ml-1m.zip and extract it
2. Download **ml-latest-small** from https://files.grouplens.org/datasets/movielens/ml-latest-small.zip (You only need `links.csv` from this to map IDs to TMDB)
3. In Colab: **Runtime → Change runtime type → T4 GPU**
4. Run **Cell 0** and upload:
   - `ratings.dat`  (from ml-1m/ folder)
   - `links.csv`    (from ml-latest-small/ folder)
5. Run all remaining cells top to bottom (~15 min on T4)
6. Cell 10 auto-downloads `embeddings.pt` and `movieid_to_tmdbid.json`

In [ ]:
# Cell 0 — Upload ml-1m ratings.dat + links.csv
#
# Click 'Choose Files' and select BOTH files at once:
#   ratings.dat   ← from ml-1m.zip
#   links.csv     ← from ml-latest-small.zip
from google.colab import files
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))
assert 'ratings.dat' in uploaded, 'Missing ratings.dat — please upload from ml-1m.zip'
assert 'links.csv'   in uploaded, 'Missing links.csv — please upload from ml-latest-small.zip'

In [ ]:
# Cell 1 — Install PyTorch Geometric
import subprocess, sys, torch

print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric'], check=True)

try:
    torch_ver = torch.__version__.split('+')[0]
    cuda_tag  = 'cu121' if torch.cuda.is_available() else 'cpu'
    pyg_url   = f'https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html'
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         'pyg_lib', 'torch_scatter', 'torch_sparse', '-f', pyg_url],
        check=True, capture_output=True
    )
    print('Compiled PyG extensions installed.')
except Exception as e:
    print(f'Optional compiled extensions skipped (pure-python fallback): {e}')

from torch_geometric.nn.models import LightGCN
import torch_geometric
print(f'torch_geometric {torch_geometric.__version__}')
assert hasattr(LightGCN, 'get_embedding'), 'get_embedding not found — update torch_geometric'
print('LightGCN.get_embedding confirmed available.')

In [ ]:
# Cell 2 — Load ml-1m ratings.dat
#
# ml-1m uses '::' as separator and has NO header row.
import pandas as pd
import numpy as np

RATINGS_PATH = 'ratings.dat'
LINKS_PATH   = 'links.csv'

# Read the ml-1m format (no header, '::' separator)
ratings = pd.read_csv(
    RATINGS_PATH,
    sep='::',
    engine='python',
    names=['userId', 'movieId', 'rating', 'timestamp'],
    dtype={'userId': int, 'movieId': int, 'rating': float, 'timestamp': int}
)
print(f'Raw: {len(ratings):,} rows | {ratings.userId.nunique():,} users | {ratings.movieId.nunique():,} movies')

# Keep implicit positives (rating >= 3.5)
ratings = ratings[ratings['rating'] >= 3.5].copy()
print(f'After rating>=3.5 filter: {len(ratings):,} rows')

# Filter sparse users/items
# Because ml-1m is already highly dense, we can easily demand 5 interactions minimum
MIN_INTERACTIONS = 5
for _ in range(3):
    u_cnt = ratings.groupby('userId')['movieId'].transform('count')
    i_cnt = ratings.groupby('movieId')['userId'].transform('count')
    ratings = ratings[(u_cnt >= MIN_INTERACTIONS) & (i_cnt >= MIN_INTERACTIONS)]

print(f'After min-{MIN_INTERACTIONS} filter: {len(ratings):,} rows | '
      f'{ratings.userId.nunique():,} users | {ratings.movieId.nunique():,} movies')

# Re-index to contiguous integers
unique_users  = sorted(ratings['userId'].unique())
unique_movies = sorted(ratings['movieId'].unique())

user2idx  = {u: i for i, u in enumerate(unique_users)}
movie2idx = {m: i for i, m in enumerate(unique_movies)}

ratings['user_idx']  = ratings['userId'].map(user2idx)
ratings['movie_idx'] = ratings['movieId'].map(movie2idx)

N_USERS  = len(unique_users)
N_ITEMS  = len(unique_movies)
print(f'\nN_USERS = {N_USERS:,}  |  N_ITEMS = {N_ITEMS:,}')
print(f'Avg ratings/user  : {len(ratings)/N_USERS:.1f}')
print(f'Avg ratings/movie : {len(ratings)/N_ITEMS:.1f}')
# The average ratings/movie will be >150, which is extremely dense and perfect for GNNs!

In [ ]:
# Cell 3 — Build bipartite edge_index tensor
import torch

# LightGCN bipartite convention:
#   user nodes →  0 .. N_USERS-1
#   item nodes →  N_USERS .. N_USERS+N_ITEMS-1
src = torch.tensor(ratings['user_idx'].values,             dtype=torch.long)
dst = torch.tensor(ratings['movie_idx'].values + N_USERS,  dtype=torch.long)

# Bidirectional edges required for symmetric message passing
edge_index = torch.stack([
    torch.cat([src, dst]),
    torch.cat([dst, src]),
], dim=0)

print(f'edge_index shape : {tuple(edge_index.shape)}')
print(f'Total graph nodes: {N_USERS + N_ITEMS:,}')

In [ ]:
# Cell 4 — Temporal 80 / 10 / 10 split
ratings_sorted = ratings.sort_values('timestamp').reset_index(drop=True)
n = len(ratings_sorted)
train_end = int(n * 0.80)
val_end   = int(n * 0.90)

train_df = ratings_sorted.iloc[:train_end]
val_df   = ratings_sorted.iloc[train_end:val_end]
test_df  = ratings_sorted.iloc[val_end:]

print(f'Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}')

# Training-only edge_index
src_tr = torch.tensor(train_df['user_idx'].values, dtype=torch.long)
dst_tr = torch.tensor(train_df['movie_idx'].values + N_USERS, dtype=torch.long)
train_edge_index = torch.stack([
    torch.cat([src_tr, dst_tr]),
    torch.cat([dst_tr, src_tr]),
], dim=0)

print(f'train_edge_index shape: {tuple(train_edge_index.shape)}')

# Per-user item sets (for negative sampling + evaluation)
def build_user_item_set(df):
    d = {}
    for uid, g in df.groupby('user_idx'):
        d[int(uid)] = set(g['movie_idx'].tolist())
    return d

train_user_items = build_user_item_set(train_df)
val_gt           = build_user_item_set(val_df)
test_gt          = build_user_item_set(test_df)

In [ ]:
# Cell 5 — Instantiate LightGCN
from torch_geometric.nn.models import LightGCN

EMBEDDING_DIM = 64
NUM_LAYERS    = 3

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model = LightGCN(
    num_nodes     = N_USERS + N_ITEMS,
    embedding_dim = EMBEDDING_DIM,
    num_layers    = NUM_LAYERS,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

train_edge_index = train_edge_index.to(device)

# Smoke-test
with torch.no_grad():
    _test_emb = model.get_embedding(train_edge_index)
print(f'get_embedding shape: {tuple(_test_emb.shape)}')
assert _test_emb.shape == (N_USERS + N_ITEMS, EMBEDDING_DIM)
print('Shape OK.')
del _test_emb

print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Cell 6 — Training loop (BPR loss)
import random
from tqdm.auto import trange

EPOCHS     = 50
BATCH_SIZE = 4096  # larger batch for ml-1m

def bpr_loss(pos_scores, neg_scores):
    return -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-8).mean()

def sample_negatives(user_batch_cpu, n_items):
    negs = []
    for uid in user_batch_cpu.tolist():
        seen = train_user_items.get(uid, set())
        while True:
            c = random.randint(0, n_items - 1)
            if c not in seen:
                negs.append(c)
                break
    return torch.tensor(negs, dtype=torch.long)

train_users_t = torch.tensor(train_df['user_idx'].values, dtype=torch.long)
train_items_t = torch.tensor(train_df['movie_idx'].values, dtype=torch.long)

losses = []

for epoch in trange(EPOCHS, desc='Training LightGCN (ml-1m)'):
    model.train()
    perm = torch.randperm(len(train_users_t))
    epoch_loss, n_batches = 0.0, 0

    for start in range(0, len(train_users_t), BATCH_SIZE):
        end     = start + BATCH_SIZE
        u_batch = train_users_t[perm[start:end]]
        p_batch = train_items_t[perm[start:end]]
        n_batch = sample_negatives(u_batch, N_ITEMS)

        u_batch = u_batch.to(device)
        p_batch = p_batch.to(device)
        n_batch = n_batch.to(device)

        all_embs  = model.get_embedding(train_edge_index)  # [N_USERS+N_ITEMS, D]
        user_embs = all_embs[:N_USERS]
        item_embs = all_embs[N_USERS:]

        u_e = user_embs[u_batch]
        p_e = item_embs[p_batch]
        n_e = item_embs[n_batch]

        pos_scores = (u_e * p_e).sum(-1)
        neg_scores = (u_e * n_e).sum(-1)

        loss = bpr_loss(pos_scores, neg_scores)

        reg = 1e-4 * (
            model.embedding.weight[u_batch].norm(2).pow(2) +
            model.embedding.weight[N_USERS + p_batch].norm(2).pow(2) +
            model.embedding.weight[N_USERS + n_batch].norm(2).pow(2)
        ) / BATCH_SIZE

        (loss + reg).backward()
        optimizer.step()
        optimizer.zero_grad()

        epoch_loss += loss.item()
        n_batches  += 1

    losses.append(epoch_loss / n_batches)
    if (epoch + 1) % 10 == 0:
        print(f'  Epoch {epoch+1:3d}  loss={losses[-1]:.4f}')

print(f'Final loss: {losses[-1]:.4f}')

In [ ]:
# Cell 7 — Evaluate Recall@20 and NDCG@20 vs popularity baseline
import math

K = 20

pop_order = (
    train_df.groupby('movie_idx')['user_idx']
    .count().sort_values(ascending=False)
    .index.tolist()
)

def recall_k(recs, gt, k):
    return len(set(recs[:k]) & gt) / min(len(gt), k)

def ndcg_k(recs, gt, k):
    dcg  = sum(1.0 / math.log2(i + 2) for i, r in enumerate(recs[:k]) if r in gt)
    idcg = sum(1.0 / math.log2(i + 2) for i in range(min(len(gt), k)))
    return dcg / idcg if idcg else 0.0

model.eval()
with torch.no_grad():
    all_embs_cpu = model.get_embedding(train_edge_index).cpu()

user_e = all_embs_cpu[:N_USERS].contiguous()
item_e = all_embs_cpu[N_USERS:].contiguous()

# Sample 1000 test users for speed
import random as _random
test_users = list(test_gt.keys())
eval_users = _random.sample(test_users, min(1000, len(test_users)))
print(f'Evaluating on {len(eval_users)} sampled test users...')

gnn_r, gnn_n, pop_r, pop_n = [], [], [], []

for uid in eval_users:
    gt = test_gt[uid]
    if not gt:
        continue
    seen = train_user_items.get(uid, set())

    scores = (user_e[uid] @ item_e.T).clone()
    if seen:
        seen_idx = torch.tensor(list(seen), dtype=torch.long)
        scores[seen_idx] = float('-inf')

    top_gnn = torch.topk(scores, k=min(K, N_ITEMS)).indices.tolist()
    top_pop = [i for i in pop_order if i not in seen][:K]

    gnn_r.append(recall_k(top_gnn, gt, K))
    gnn_n.append(ndcg_k(top_gnn, gt, K))
    pop_r.append(recall_k(top_pop, gt, K))
    pop_n.append(ndcg_k(top_pop, gt, K))

print(f'Evaluated on {len(gnn_r)} users')
print(f'LightGCN   Recall@{K}: {np.mean(gnn_r):.4f}  NDCG@{K}: {np.mean(gnn_n):.4f}')
print(f'Popularity Recall@{K}: {np.mean(pop_r):.4f}  NDCG@{K}: {np.mean(pop_n):.4f}')

In [ ]:
# Cell 8 — Plot training loss curve
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 4))
plt.plot(range(1, len(losses) + 1), losses, marker='o', markersize=3,
         linewidth=1.5, color='#7c3aed')
plt.xlabel('Epoch')
plt.ylabel('BPR Loss')
plt.title('LightGCN Training Loss — ml-1m')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 9 — Export embeddings.pt and movieid_to_tmdbid.json
import json

model.eval()
with torch.no_grad():
    final_embs = model.get_embedding(train_edge_index).cpu()

item_embeddings = final_embs[N_USERS:].contiguous()   # [N_ITEMS, D]
torch.save(item_embeddings, 'embeddings.pt')
print(f'Saved embeddings.pt  shape={tuple(item_embeddings.shape)}')

# Build movieid_to_tmdbid.json using links.csv from ml-latest-small
links = pd.read_csv(LINKS_PATH, dtype=str)
links = links.dropna(subset=['tmdbId'])
links = links[links['tmdbId'].str.strip().str.isdigit()]
tmdb_lookup = {row['movieId'].strip(): int(row['tmdbId']) for _, row in links.iterrows()}

tmdb_map = {
    str(m): tmdb_lookup[str(m)]
    for m in unique_movies   # sorted list from Cell 2
    if str(m) in tmdb_lookup
}

with open('movieid_to_tmdbid.json', 'w') as f:
    json.dump(tmdb_map, f, separators=(',', ':'))
print(f'Saved movieid_to_tmdbid.json  entries={len(tmdb_map)}')


In [ ]:
# Cell 10 — Download both output files
from google.colab import files

print('Downloading embeddings.pt ...')
files.download('embeddings.pt')

print('Downloading movieid_to_tmdbid.json ...')
files.download('movieid_to_tmdbid.json')

print()
print('=' * 60)
print('DONE! Place both files inside backend/:')
print('  backend/embeddings.pt')
print('  backend/movieid_to_tmdbid.json')
print('=' * 60)
print()